In [4]:
import glob, os
import geopandas as gpd
import rasterio

In [5]:
# read the geopandas dataframe
gdf_infile = r'../resources/india_utm_fishnet.gpkg'
gdf = gpd.read_file(gdf_infile)

for input_tiff in sorted(glob.glob('../output/flood_raster/*.tif')):
    print(f'Processing {input_tiff}...')
    
    # Input and output file paths
    directory, filename = os.path.split(input_tiff)
    output_tiff = os.path.join(directory, 'reprojected', filename.replace('.tif', '_reprojected.tif'))

    # Define the correct projection (in this case, WGS 84)
    tile_id = int(os.path.splitext(input_tiff)[0].split('_')[-1])
    zone = int(gdf.loc[gdf.ID == tile_id].zone.values[0][:2])
    new_crs = f'EPSG:326{zone}'
    
    # Open the input GeoTIFF file and get its metadata
    with rasterio.open(input_tiff) as src:
        # Get band names
        band_names = src.descriptions

        metadata = src.meta.copy()
        metadata.update({
            'crs': new_crs
        })

        # Create a new GeoTIFF file with the correct projection
        with rasterio.open(output_tiff, 'w', **metadata) as dst:
            # Write the data from the input raster to the output raster
            for i in range(1, len(band_names) + 1):
                band_data = src.read(i)
                dst.write(band_data, i)

                # Set band names
                dst.set_band_description(i, band_names[i - 1])



Processing ../output/flood_raster/floodextentstacked_2018_2022_148.tif...
Processing ../output/flood_raster/floodscenescount_2018_2022_148.tif...


## compress the folder so that it is easier to download

In [3]:
import zipfile
import os

def zip_folder(folder_path, zip_filename):
    # Create a Zip file
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Walk through all the files in the folder
        for root, _, files in os.walk(folder_path):
            for file in files:
                # Create the full file path
                file_path = os.path.join(root, file)
                # Add the file to the Zip file
                zipf.write(file_path, os.path.relpath(file_path, folder_path))

# Example usage
folder_to_zip = '../output/flood_raster/reprojected/'  # Replace this with the path to your folder
zip_file_name = '../output/flood_raster/flood_raster_reprojected.zip'  # Name for the zip file

zip_folder(folder_to_zip, zip_file_name)
print(f'Folder "{folder_to_zip}" has been compressed to "{zip_file_name}"')

Folder "../output/flood_raster/reprojected/" has been compressed to "../output/flood_raster/flood_raster_reprojected.zip"
